# Remunerômetro — RAIS 2025 | Aprendizagem de Máquina Supervisionada

**UFPB · atividade individual — relatório computacional reproduzível**

**Problema aplicado.** Estimar a remuneração nominal de dezembro de 2025 dos vínculos formais ativos em 31/12 nas 35 ocupações selecionadas. Comparar duas especificações: **A — três grupos profissionais** e **B — 35 CBOs individualizados**. Ambas usam os mesmos atributos adicionais e a mesma partição de dados.

> **Escopo deste notebook:** documenta a preparação, demonstra o pré-processamento, recupera resultados reais produzidos nos scripts 04 e 05 e apresenta diagnóstico e conclusão. **Não reabre o teste e não ajusta hiperparâmetros.** Para repetir o treinamento integral, executar os scripts 04 e 05 na ordem documentada, usando os artefatos e os mesmos dados. Para a entrega acadêmica, confrontar a organização com o template obrigatório e o laboratório 5.


## 1. Fontes, universo e unidade estatística

Fonte: RAIS pública 2025, registros individuais de **vínculo formal** (não de pessoas únicas). Seleção de 35 CBOs em três grupos: Estatística formal e quantitativas, TI e dados, Expansão profissional. Elegibilidade: indicador de vínculo ativo em 31/12 igual a 1 e remuneração nominal de dezembro positiva. A tipologia IBGE refere-se ao **município do estabelecimento empregador**, não à residência ou necessariamente ao local de teletrabalho.

O fluxo oficial de preparação é `01_preparar_rais.py` → `03_preparar_modelagem.py` → `02_auditar_base.py` (no GitHub os nomes podem variar). O arquivo de modelagem fica **fora do repositório público**, em `C:/Users/uriel/data/analysis/remunerometro/v3/modelagem_dupla/rais_2025_base_modelagem_dezembro.parquet`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DADOS = Path(r"C:\Users\uriel\data\analysis\remunerometro\v3\modelagem_dupla")
TREINO = DADOS / "treinamento"
AVALIACAO = DADOS / "avaliacao_final"

# Opcional: copiar os relatórios agregados para "notebooks/artefatos/"
# e executar o notebook sem microdados ou caminhos locais.
REPO = Path.cwd()
ARTEFATOS = REPO / "notebooks" / "artefatos"

def localizar(nome: str, pasta: Path) -> Path:
    preferencia = pasta / nome
    alternativa = ARTEFATOS / nome
    if preferencia.exists():
        return preferencia
    if alternativa.exists():
        return alternativa
    raise FileNotFoundError(f"Arquivo {nome!r} não encontrado em {preferencia} ou {alternativa}")

def ler_csv(nome: str, pasta: Path) -> pd.DataFrame:
    return pd.read_csv(localizar(nome, pasta), sep=";", decimal=",000")


## 2. Qualidade e preparação dos atributos

A variável-alvo é `ln_rem_dez = ln(rem_dez_nom)`; sua validade depende de remuneração positiva. As categorias ocupacionais são codificadas por one-hot **dentro de `Pipeline`**, evitando aprender categorias ou valores de imputação a partir do teste. Os preditores são `uf_sigla`, `tipologia_ibge`, `escolaridade`, `cnae_divisao`, `idade`, `horas` e **um** atributo ocupacional (grupo *ou* CBO). A remuneração média anual não é preditora.

A função abaixo ilustra exatamente o desenho do pré-processamento do script 04; não constitui um novo ajuste de modelos.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAS = ["uf_sigla", "tipologia_ibge", "escolaridade", "cnae_divisao"]
NUMERICAS = ["idade", "horas"]

def pre_processador(variavel_ocupacional: str):
    cat = Pipeline([
        ("imputar", SimpleImputer(strategy="constant", fill_value="Ausente")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=True)),
    ])
    num = Pipeline([
        ("imputar", SimpleImputer(strategy="median", add_indicator=True)),
        ("escalar", StandardScaler()),
    ])
    return ColumnTransformer([
        ("categoricas", cat, [variavel_ocupacional] + CATEGORICAS),
        ("numericas", num, NUMERICAS),
    ], sparse_threshold=0.3)

print("Modelo A:", ["grupo_ocupacional"] + CATEGORICAS + NUMERICAS)
print("Modelo B:", ["cbo"] + CATEGORICAS + NUMERICAS)


## 3. Partição, amostra e validação cruzada

O script 04 reservou **20% da população elegível para teste** (sem acessar o desfecho para escolher modelos), sorteou **2.000 vínculos exclusivamente do treino** com semente **5327** e realizou KFold com **5 dobras**. Em ambas as especificações, foram comparados baseline (`DummyRegressor`), MQO, Ridge e Lasso, com grade de regularização entre 10⁻⁴ e 10⁴.

**Verificação obrigatória:** a proporção do teste deve corresponder à especificação do laboratório 5. A semente informada segue o que consta nos arquivos da execução; a conformidade com os seis dígitos da matrícula é responsabilidade do autor. Métricas: MAE em R$ e RMSE na escala log.

In [ ]:
meta = json.loads(localizar("metadados_treino.json", TREINO).read_text(encoding="utf-8"))
print({k: meta[k] for k in ["semente", "frac_teste", "n_populacao", "n_treino_populacao", "n_teste_intocado", "n_amostra", "dobras"]})
print("Modelos selecionados:", meta["selecionados_cv"])


## 4. Comparação por validação cruzada (sem consultar o teste)

In [ ]:
cv = ler_csv("comparacao_validacao_cruzada.csv", TREINO)
cols = ["especificacao", "algoritmo", "alpha", "mae_treino", "mae_cv_media", "mae_cv_dp", "rmse_log_cv_media"]
display(cv[cols].sort_values(["especificacao", "mae_cv_media"]).reset_index(drop=True))
fig, ax = plt.subplots(figsize=(10, 4))
for nome, grupo in cv.groupby("especificacao"):
    x = grupo.sort_values("mae_cv_media").head(4)
    ax.scatter([f"{nome}: {a}" for a in x["algoritmo"]], x["mae_cv_media"], label=nome)
ax.set_ylabel("MAE na validação cruzada (R$)")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.show()


## 5. Curvas de validação dos modelos penalizados

Esta visualização utiliza a grade de alphas escolhida na etapa 04; não escolhe novos hiperparâmetros depois do teste.

In [ ]:
curvas = ler_csv("curvas_validacao.csv", TREINO)
fig, ax = plt.subplots(figsize=(9, 5))
for (modelo, algoritmo), grupo in curvas.groupby(["especificacao", "algoritmo"]):
    if algoritmo in ("lasso", "ridge"):
        ax.plot(grupo["alpha"], grupo["mae_cv_media"], marker="o", label=f"{modelo} — {algoritmo}")
ax.set_xscale("log")
ax.set_xlabel("Alpha (escala log)")
ax.set_ylabel("MAE médio em 5 dobras (R$)")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()


## 6. Complexidade e seleção das especificações

O arquivo do treinamento registra **p=110** no modelo por famílias e **p=137** no modelo por CBO. Com n=2.000, temos `p/n = 5,5%` e `6,85%`. O maior detalhamento de CBO eleva a dimensionalidade e pode produzir estimativas frágeis para ocupações raras. A seleção foi realizada pelo menor MAE médio em validação: Lasso com `alpha=0,001` para cada especificação. Não se deve inferir causalidade dos coeficientes.


In [ ]:
selecionados = meta["selecionados_cv"]
display(pd.DataFrame([
    {"Especificação": nome, "Algoritmo": v["algoritmo"], "Alpha": v["alpha"],
     "p": v["p"], "n": v["n"], "p/n": v["p_sobre_n"], "MAE CV (R$)": v["mae_cv"]}
    for nome, v in selecionados.items()
]))


## 7. Avaliação FINAL no teste intocado — resultados congelados

**Esta seção apenas lê `01_metricas_finais.csv`, produzido pelo script 05.** Não carrega `indices_particao.npz`, não altera os modelos e não calcula novas métricas para selecionar alternativas.

A avaliação foi concluída uma vez sobre **234.693 vínculos de teste**, após escolha por CV. O desempenho é descritivo e não constitui autorização para retreinar usando o teste.

In [ ]:
final = ler_csv("01_metricas_finais.csv", AVALIACAO)
mostra = ["especificacao", "algoritmo_congelado_cv", "alpha_congelado_cv", "n_teste", "mae_cv_2mil_reais", "mae_teste_reais", "rmse_teste_log", "vies_medio_reais_previsto_menos_observado", "p_sobre_n_amostra"]
display(final[mostra])
fig, ax = plt.subplots(figsize=(7, 4))
final.plot(x="especificacao", y=["mae_cv_2mil_reais", "mae_teste_reais"], kind="bar", ax=ax)
ax.set_ylabel("MAE (R$)")
ax.set_xlabel("Especificação")
ax.legend(["Validação cruzada", "Teste"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Diagnósticos de erro por sexo e escolaridade

As diferenças descritivas de erros entre grupos **não identificam discriminação causal**. Categorias pouco representadas exigem cautela; a escala salarial de cada grupo também altera o MAE absoluto.

In [ ]:
# As tabelas completas foram produzidas na etapa 05.
for nome in ["03_erros_por_sexo.csv", "03_erros_por_escolaridade.csv", "03_erros_por_grupo_ocupacional.csv"]:
    try:
        print("\n", nome)
        display(ler_csv(nome, AVALIACAO).head(30))
    except FileNotFoundError:
        print(f"Arquivo opcional ausente: {nome}. Copie-o de avaliacao_final/ para notebooks/artefatos/.")


## 9. Interpretação dos resultados e limitações

A avaliação final registrou **MAE de R&#36; 4.338,33** e **RMSE(log)=0,6171** para o Modelo A (famílias); o Modelo B (35 CBOs) obteve **MAE de R&#36; 4.115,23** e **RMSE(log)=0,5865**. A diferença de aproximadamente **R&#36; 223,11**, favorável ao modelo detalhado, é descritiva e não deve motivar nova seleção após exposição ao teste.

As médias observadas no teste foram de aproximadamente R&#36; 9.230,74, enquanto as médias previstas foram R&#36; 7.672,44 e R&#36; 7.784,68, respectivamente. O viés médio negativo pode ser explicado em parte por prever o logaritmo e converter por exponencial: $E[\log Y \mid X]$ não é necessariamente $E[Y \mid X]$. Previsões em áreas e CBOs raros são menos estáveis. Os dados da RAIS não representam renda de autônomos e PJ. O município identifica o estabelecimento, não necessariamente a residência. O dashboard público agrega o teste por território × CBO e suprime células com n<30, de modo que seus totais podem diferir do teste completo.


## 10. Reprodutibilidade, uso de IA e próximos componentes

Os artefatos de rastreabilidade incluem `metadados_treino.json`, `indices_particao.npz`, dois pipelines congelados e `AVALIACAO_FINAL_CONCLUIDA.json` com hashes. O dashboard **local** trabalha com estatísticas municipais internas; o dashboard **público** utiliza agregados em `data_public/`.

**Declaração de apoio de IA:** ferramentas de IA foram utilizadas como apoio na elaboração de scripts, revisão de código, organização de documentação e interpretação inicial de diagnósticos. O autor é responsável pelas escolhas metodológicas, validação das saídas, execução, interpretação e integridade acadêmica da entrega.
